# Inference

This continues from [the deployment notebook](https://github.com/RobustIntelligence/foundation-ai-cookbook/blob/main/3_adoptions/deployment/sagemaker/foundation_sec_8b_reasoning/deploy.ipynb). <br>
Let's perform inference using the reasoning model deployed on Amazon SageMaker AI. <br>
As a prerequisite, please ensure you have access to SageMaker from the environment where this notebook is being run.

Since this is a reasoning model, the response will contain a `reasoning_content` field with the chain-of-thought
and a `content` field with the final answer.

In [ ]:
import boto3
import json

In [ ]:
sagemaker_runtime = boto3.client('sagemaker-runtime')
endpoint_name = ''  # copy & paste an endpoint created by the deployment notebook
assert endpoint_name, "Set endpoint_name above before running inference"

# Reasoning models produce more tokens due to chain-of-thought, so set a higher limit
max_tokens = 8192

In [ ]:
def chat_generation(user_prompt):
    messages = [
        {
            "role": "user",
            "content": user_prompt
        }
    ]
    payload = {
        "messages": messages,
        "temperature": 0.3,
        "max_tokens": max_tokens,
    }
    response = sagemaker_runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        Body=json.dumps(payload),
        ContentType="application/json",
    )
    body = json.loads(response['Body'].read().decode('utf-8'))
    message = body["choices"][0]["message"]

    reasoning = message.get("reasoning_content", "")
    answer = message.get("content", "")
    return reasoning, answer

In [ ]:
user_prompt = "What is cybersecurity?"

reasoning, answer = chat_generation(user_prompt)

print("=== Reasoning (chain-of-thought) ===")
print(reasoning)
print()
print("=== Answer ===")
print(answer)